
# Phase 1 — Task 1: Data Audit and Exploration

## Objective
Inspect the CortexHR employee attrition dataset, identify data-quality issues, and document decisions before cleaning or model training.

## Dataset
- Workbook: `02_CortexHR_AI_ML_Dataset.xlsx`
- Main sheet: `attrition_dataset`
- Records: 2,389
- Columns: 42

## Audit checklist
1. Dataset size and data types
2. Missing values
3. Attrition class distribution
4. Exact duplicate rows and repeated employee IDs
5. Numeric outliers and impossible values
6. Inconsistent categorical labels
7. Potential target leakage
8. Data-quality decisions

**Important:** The original workbook will remain unchanged.


In [1]:


from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Find the project folder even if the notebook starts elsewhere
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "raw" / "02_CortexHR_AI_ML_Dataset.xlsx").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "02_CortexHR_AI_ML_Dataset.xlsx"
)

print("Project folder:", PROJECT_ROOT)
print("Excel file found:", DATA_PATH.exists())
print("Excel path:", DATA_PATH)


Project folder: c:\Users\ashwitha\OneDrive\Documents\growmar-phase1
Excel file found: True
Excel path: c:\Users\ashwitha\OneDrive\Documents\growmar-phase1\data\raw\02_CortexHR_AI_ML_Dataset.xlsx


In [2]:

import pandas as pd
import numpy as np

df = pd.read_excel(
    DATA_PATH,
    sheet_name="attrition_dataset"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


Dataset loaded successfully!
Shape: (2389, 42)


,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,...,paid_leave_days_6m,dept_attrition_rate_prior_12m,left_within_12m,voluntary_left_within_12m,source,exit_date,resignation_date,exit_reason,exit_interview_done,fnf_settlement_amount
0,1,EMP-0002,2025-09-30,D08,R047,4,Pune,Remote,Permanent,B.E. Computer Science,...,5,0.2353,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
1,2,EMP-0004,2025-09-30,D03,R017,3,Erode,Hybrid,Permanent,MCA,...,4,0.0556,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
2,3,EMP-0011,2025-09-30,D07,R039,1,Kochi,Hybrid,Permanent,M.A. Social Work,...,7,0.0000,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
3,4,EMP-0012,2025-09-30,D01,R004,2,Chennai,Onsite,Permanent,B.Tech Information Technology,...,8,0.1448,1,1,master_employee_snapshot,2026-04-23,2026-03-24,Work-Life Balance,Y,117098.85
4,5,EMP-0013,2025-09-30,D04,R023,2,Chennai,Onsite,Permanent,B.E. Mechanical,...,7,0.2051,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN


In [3]:

# Check missing values in every column

missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values(
    by="missing_percent",
    ascending=False
)

print("Columns containing missing values:")
display(missing_summary)


Columns containing missing values:


,missing_count,missing_percent
resignation_date,2068,86.56
exit_date,2040,85.39
exit_reason,2040,85.39
fnf_settlement_amount,2040,85.39
exit_interview_done,2040,85.39
months_since_last_promotion,1459,61.07
last_engagement_score,528,22.10
last_raise_pct,202,8.46
last_performance_rating,182,7.62
commute_distance_km,55,2.30


In [4]:

# Check the attrition target distribution

target_column = "left_within_12m"

target_counts = df[target_column].value_counts().sort_index()
target_percentages = (
    df[target_column]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "employee_count": target_counts,
    "percentage": target_percentages
})

target_summary.index.name = "left_within_12m"

print("Target distribution:")
display(target_summary)

print("\nInterpretation:")
print("0 = Employee stayed")
print("1 = Employee left within 12 months")


Target distribution:


,employee_count,percentage
left_within_12m,,
0,2040,85.39
1,349,14.61



Interpretation:
0 = Employee stayed
1 = Employee left within 12 months


In [5]:

# Check exact duplicate rows and repeated employee IDs

print("Exact duplicate rows:", df.duplicated().sum())

repeated_ids = df[
    df.duplicated(subset=["employee_id"], keep=False)
].sort_values("employee_id")

print(
    "\nNumber of rows with repeated employee IDs:",
    len(repeated_ids)
)

print(
    "Number of distinct repeated employee IDs:",
    repeated_ids["employee_id"].nunique()
)

display(
    repeated_ids[
        [
            "row_id",
            "employee_id",
            "observation_date",
            "left_within_12m",
            "voluntary_left_within_12m"
        ]
    ].sort_values(["employee_id", "row_id"])
)


Exact duplicate rows: 0

Number of rows with repeated employee IDs: 12
Number of distinct repeated employee IDs: 6


,row_id,employee_id,observation_date,left_within_12m,voluntary_left_within_12m
26,27,EMP-0060,2025-09-30,0,0
2386,2387,EMP-0060,2025-09-30,0,0
100,101,EMP-0237,2025-09-30,0,0
2387,2388,EMP-0237,2025-09-30,0,0
101,102,EMP-0238,2025-09-30,0,0
2384,2385,EMP-0238,2025-09-30,0,0
150,151,EMP-0325,2025-09-30,0,0
2383,2384,EMP-0325,2025-09-30,0,0
250,251,EMP-0483,2025-09-30,0,0
2385,2386,EMP-0483,2025-09-30,0,0


In [6]:

# Validate numeric values against expected ranges

range_checks = {
    "tenure_months": (0, None),
    "total_experience_years": (0, None),
    "commute_distance_km": (0, None),
    "attendance_rate_3m": (0, 1),
    "wfh_share_3m": (0, 1),
}

for column, (lower, upper) in range_checks.items():
    values = df[column]

    invalid = pd.Series(False, index=df.index)

    if lower is not None:
        invalid |= values < lower

    if upper is not None:
        invalid |= values > upper

    print(f"\n{column}: {int(invalid.sum())} values outside expected range")

    if invalid.any():
        display(
            df.loc[invalid, ["employee_id", column]]
            .head(10)
        )



tenure_months: 4 values outside expected range


,employee_id,tenure_months
108,EMP-0248,-4
125,EMP-0280,-5
312,EMP-0568,-2
339,EMP-0596,-1



total_experience_years: 10 values outside expected range


,employee_id,total_experience_years
516,HIST-0134,-0.3
871,HIST-0489,-0.2
1105,HIST-0723,-0.7
1650,HIST-1268,-0.1
1814,HIST-1432,-0.8
1818,HIST-1436,-1.1
1945,HIST-1563,-0.2
2154,HIST-1772,-0.4
2170,HIST-1788,-0.7
2362,HIST-1980,-0.2



commute_distance_km: 0 values outside expected range

attendance_rate_3m: 2 values outside expected range


,employee_id,attendance_rate_3m
430,HIST-0048,1.12
1233,HIST-0851,1.12



wfh_share_3m: 213 values outside expected range


,employee_id,wfh_share_3m
386,HIST-0004,-0.035
390,HIST-0008,-0.036
398,HIST-0016,-0.051
399,HIST-0017,-0.031
429,HIST-0047,-0.063
443,HIST-0061,-0.016
447,HIST-0065,-0.021
469,HIST-0087,-0.002
483,HIST-0101,-0.050
492,HIST-0110,-0.026


In [7]:

# Investigate inconsistent categorical labels

categorical_columns = [
    "work_location",
    "work_mode",
    "employment_type",
    "highest_education",
]

for column in categorical_columns:
    print(f"\n--- {column} ---")

    print("Number of unique values:", df[column].nunique(dropna=False))

    print(
        df[column]
        .value_counts(dropna=False)
        .to_string()
    )



--- work_location ---
Number of unique values: 16
work_location
Bengaluru     888
Chennai       443
Hyderabad     292
Pune          228
Mumbai        155
Delhi NCR     137
Coimbatore    129
Kochi          53
Erode          51
BENGALURU       3
Bangalore       3
bengaluru       2
chennai         2
delhi ncr       1
Delhi-NCR       1
Madras          1

--- work_mode ---
Number of unique values: 3
work_mode
Hybrid    1107
Onsite     829
Remote     453

--- employment_type ---
Number of unique values: 2
employment_type
Permanent              2239
Fixed-Term Contract     150

--- highest_education ---
Number of unique values: 28
highest_education
B.Tech Information Technology    244
M.Tech                           233
B.E. Computer Science            229
MCA                              210
MBA                              198
B.Com                            178
BBA                              173
B.Sc Computer Science            166
B.E. Electronics                 124
B.Sc            

In [8]:

for column in ["work_mode", "employment_type"]:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).to_string())

print("\n--- Education categories ---")
print(df["highest_education"].value_counts(dropna=False).to_string())

print("\n--- Location variants to investigate ---")
print(
    df.loc[
        df["work_location"].astype("string").str.lower().isin(
            ["bengaluru", "bangalore", "chennai", "madras", "delhi ncr", "delhi-ncr"]
        ),
        "work_location"
    ].value_counts(dropna=False).to_string()
)



--- work_mode ---
work_mode
Hybrid    1107
Onsite     829
Remote     453

--- employment_type ---
employment_type
Permanent              2239
Fixed-Term Contract     150

--- Education categories ---
highest_education
B.Tech Information Technology    244
M.Tech                           233
B.E. Computer Science            229
MCA                              210
MBA                              198
B.Com                            178
BBA                              173
B.Sc Computer Science            166
B.E. Electronics                 124
B.Sc                             110
B.A.                              72
B.E. Mechanical                   70
Diploma Engineering               60
M.Sc Statistics                   44
B.Des                             35
M.A. Social Work                  28
MBA Finance                       23
M.Com                             22
CA                                21
CMA                               21
B.A. Mass Communication           20
LLM 


## Category Investigation — Findings

- `work_mode`: 3 consistent categories — Hybrid (1,107), Onsite (829), and Remote (453).
- `employment_type`: 2 categories — Permanent (2,239) and Fixed-Term Contract (150).
- `work_location`: 16 unique labels include spelling and capitalization variants. Equivalent city labels should be standardized through a documented mapping.
- `highest_education`: Multiple distinct qualifications are present, with 15 missing values. Preserve distinct qualifications unless equivalence is confirmed.

## Audit Conclusion

The dataset contains 2,389 records and 42 columns. The audit identified missing values, repeated employee IDs, invalid numeric values, potential outliers, inconsistent city labels, and possible target leakage.

No original data has been deleted or modified. Cleaning decisions will be applied to a separate processed dataset after validation.
